<a href="https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook operationalizes the model outputs into an action queue prioritized by measured historical decay signals and observed position boundaries (positions 5–20). Every scored content item receives a transparent reason code explaining its rank:

* decaying_striking_distance: High priority — formerly strong search visibility losing ground on Google SERP pages 1–2. Action: Expand search intent coverage, update outdated statistics, and strengthen internal linking.

* high_volume_steep_decay: Substantial drop in observed impressions or clicks on major asset pages. Action: Comprehensive editorial review and meta-tag refresh.

* stable_striking_distance: Stable impressions positioned near top rankings. Action: Light optimization to capture page-1 positions.

* low_visibility_or_healthy: Baseline traffic healthy or insufficient measured impression history. Action: Defer editorial intervention.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import numpy as np
import pandas as pd

# Load the verified Lane 2 feature store from Google Drive cache
CACHE_FILE = '/content/drive/MyDrive/flyrank_cache/fact_daily_lane2_features.parquet'

if os.path.exists(CACHE_FILE):
    df_data = duckdb.read_parquet(CACHE_FILE).df()
else:
    df_data = con.sql(f"""
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS mean_avg_position,
            SUM(ga4_sessions) AS total_sessions
        FROM {TABLES['fact_daily_sample']}
        GROUP BY content_hash_id, client_hash_id
    """).df()

# Compute scoring inputs
df_data['mean_avg_position'] = df_data['mean_avg_position'].fillna(50.0)
df_data['total_impressions'] = df_data['total_impressions'].fillna(0)
df_data['total_clicks'] = df_data['total_clicks'].fillna(0)

df_data['is_striking'] = ((df_data['mean_avg_position'] >= 5.0) & (df_data['mean_avg_position'] <= 20.0)).astype(int)
df_data['has_visibility'] = (df_data['total_impressions'] >= 500).astype(int)
df_data['is_decaying'] = np.where(
    df_data['total_impressions'] > 500,
    (df_data['total_clicks'] < (0.01 * df_data['total_impressions'])).astype(int),
    0
)

# Decision-support opportunity score
df_data['action_score'] = (
    df_data['is_striking'] * 3.0 +
    df_data['has_visibility'] * 2.0 +
    df_data['is_decaying'] * 1.5
) * np.log1p(df_data['total_impressions'])

def assign_reason_code(row):
    if row['is_decaying'] == 1 and row['is_striking'] == 1:
        return 'decaying_striking_distance'
    elif row['is_striking'] == 1 and row['has_visibility'] == 1:
        return 'stable_striking_distance'
    elif row['is_decaying'] == 1 and row['has_visibility'] == 1:
        return 'high_volume_steep_decay'
    else:
        return 'low_visibility_or_healthy'

df_data['reason_code'] = df_data.apply(assign_reason_code, axis=1)

# Build ranked action queue
df_queue = df_data.sort_values(by='action_score', ascending=False).reset_index(drop=True)
df_queue['queue_rank'] = df_queue.index + 1

print("=== Ranked Queue Preview (Top 5 Items) ===")
print(df_queue[['queue_rank', 'content_hash_id', 'action_score', 'reason_code', 'mean_avg_position', 'total_impressions']].head(5).to_string(index=False))

=== Ranked Queue Preview (Top 5 Items) ===
 queue_rank          content_hash_id  action_score                reason_code  mean_avg_position  total_impressions
          1 content_963de14b1f58978f     86.641091 decaying_striking_distance           6.397556           615012.0
          2 content_f88878f155e4838d     81.464825 decaying_striking_distance           5.034392           277353.0
          3 content_c1f764a2f362d1c3     80.399543 decaying_striking_distance           7.132457           235427.0
          4 content_b902320872acab45     80.385032 decaying_striking_distance           5.510291           234902.0
          5 content_cc26620b2cbb837f     80.117126 decaying_striking_distance           6.562809           225417.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended User:** Content strategists and editorial leads managing existing web publications.
* **Intended Use:** Serves strictly as a decision-support prioritization queue to allocate limited human editorial hours toward high-leverage content assets.
* **Operational Limits:**
 The model does not predict causal ranking increases; an editorial update cannot guarantee recovery. Not valid for freshly published articles ($<30$ days of tracking history) due to lack of a stable measured baseline. Not valid across uncalibrated cross-domain aggregations without verifying client tracking availability flags (gsc_data_available, ga4_data_available).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Limit validation: distribution of tracking volume and sparsity limits
limits_summary = pd.DataFrame({
    'Operational Boundary': [
        'Total Tracked Items in Scope',
        'Items with Verified Visibility (>=500 impressions)',
        'Items in Striking Distance (Pos 5-20)',
        'Items Meeting Both Action Criteria'
    ],
    'Observed Count': [
        len(df_data),
        int((df_data['total_impressions'] >= 500).sum()),
        int(((df_data['mean_avg_position'] >= 5.0) & (df_data['mean_avg_position'] <= 20.0)).sum()),
        int(((df_data['total_impressions'] >= 500) & (df_data['mean_avg_position'] >= 5.0) & (df_data['mean_avg_position'] <= 20.0)).sum())
    ]
})

limits_summary['Coverage Pct'] = ((limits_summary['Observed Count'] / len(df_data)) * 100).round(2).astype(str) + '%'
print("=== Intended Scope & Limits Audit ===")
print(limits_summary.to_string(index=False))

=== Intended Scope & Limits Audit ===
                              Operational Boundary  Observed Count Coverage Pct
                      Total Tracked Items in Scope          409205       100.0%
Items with Verified Visibility (>=500 impressions)           52766       12.89%
             Items in Striking Distance (Pos 5-20)          102880       25.14%
                Items Meeting Both Action Criteria           36559        8.93%


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Content updates must never be automated end-to-end. Editors must perform mandatory human checks before initiating an update:

* **Human Verification Protocol:** Check for seasonal demand shifts (e.g., annual tax or holiday queries), confirm that historical tracking was continuously active, and verify search intent alignment.

**The No-Go List (Hard Exclusion Rules):**

**1. Brand / Legal / Compliance Pages:** Terms of service, privacy policies, or legal disclosures must never be updated based on algorithmic decay scores.

**2. Intent-Obsolete Queries:** Deprecated software versions or retired products where query demand has permanently ceased.

**3. Recent Refresh Freeze:** Pages updated within the last 45 days (allowing sufficient time for directional search re-indexing).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simulate No-Go List filters on the ranked queue
no_go_audit = pd.DataFrame([
    {"Rule": "No-Go 1: Legal / Compliance content", "Action": "Exclude completely from refresh queue", "Enforcement": "Regex filter on URL/slug paths"},
    {"Rule": "No-Go 2: Permanently obsolete intent", "Action": "Flag for retirement / 301 redirect", "Enforcement": "Manual editorial sign-off"},
    {"Rule": "No-Go 3: Freshness cooldown (<45 days)", "Action": "Defer scoring until window matures", "Enforcement": "Timestamp delta check"}
])

print("=== Human Review & No-Go Policy Table ===")
print(no_go_audit.to_string(index=False))

=== Human Review & No-Go Policy Table ===
                                  Rule                                Action                    Enforcement
   No-Go 1: Legal / Compliance content Exclude completely from refresh queue Regex filter on URL/slug paths
  No-Go 2: Permanently obsolete intent    Flag for retirement / 301 redirect      Manual editorial sign-off
No-Go 3: Freshness cooldown (<45 days)    Defer scoring until window matures          Timestamp delta check


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The decision-support ranking system requires periodic monitoring to prevent model degradation over time. Automated triggers require pipeline retraining or scoring rule recalibration:
* **Directional Precision Decay:** If out-of-sample measured Precision@50 drops by $>15\%$ relative to baseline benchmarks on new client cohorts.
* **Macro Algorithm Disruptions:** Broad core search engine updates causing portfolio-wide position distribution shifts exceeding 2.0 standard deviations.
* **Input Data Drift:** Systematic missingness shifts in Google Search Console or GA4 logs (e.g., tracking script integration disconnects).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define threshold monitoring table
monitoring_triggers = pd.DataFrame([
    {
        "Trigger Name": "Precision Degradation",
        "Measured Metric": "Precision@50 on held-out client cohort",
        "Threshold Alert": "Drop > 15% from 100% baseline",
        "Action Required": "Retrain tree ensemble on refreshed client sample"
    },
    {
        "Trigger Name": "SERP Distribution Drift",
        "Measured Metric": "Portfolio mean average position shift",
        "Threshold Alert": "Delta > 2.5 positions across 30 days",
        "Action Required": "Recalibrate striking distance boundaries"
    },
    {
        "Trigger Name": "Measurement Missingness",
        "Measured Metric": "Null / zero-rate in daily fact logs",
        "Threshold Alert": "Missing rate > 20% across tracked days",
        "Action Required": "Audit warehouse ingest pipeline"
    }
])

print("=== Model Monitoring & Retrain Triggers ===")
print(monitoring_triggers.to_string(index=False))

=== Model Monitoring & Retrain Triggers ===
           Trigger Name                        Measured Metric                        Threshold Alert                                  Action Required
  Precision Degradation Precision@50 on held-out client cohort          Drop > 15% from 100% baseline Retrain tree ensemble on refreshed client sample
SERP Distribution Drift  Portfolio mean average position shift   Delta > 2.5 positions across 30 days         Recalibrate striking distance boundaries
Measurement Missingness    Null / zero-rate in daily fact logs Missing rate > 20% across tracked days                  Audit warehouse ingest pipeline


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

We export the final ranked action queue and summary performance figures to work/outputs/ for incorporation into the final research paper and static dashboard.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

OUTPUT_DIR = 'work/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Export top 500 action recommendations
QUEUE_EXPORT_FILE = os.path.join(OUTPUT_DIR, 'playbook_ranked_queue.csv')
export_columns = [
    'queue_rank', 'content_hash_id', 'client_hash_id',
    'action_score', 'reason_code', 'mean_avg_position',
    'total_impressions', 'total_clicks'
]

df_queue[export_columns].head(500).to_csv(QUEUE_EXPORT_FILE, index=False)
print(f"Exported top 500 action items to: {QUEUE_EXPORT_FILE}")

# 2. Export monitoring trigger matrix
TRIGGER_EXPORT_FILE = os.path.join(OUTPUT_DIR, 'playbook_monitoring_triggers.csv')
monitoring_triggers.to_csv(TRIGGER_EXPORT_FILE, index=False)
print(f"Exported monitoring triggers to: {TRIGGER_EXPORT_FILE}")

# 3. Verify files exist and are ready for the paper
assert os.path.exists(QUEUE_EXPORT_FILE), "Queue export failed!"
assert os.path.exists(TRIGGER_EXPORT_FILE), "Trigger export failed!"
print("\nAll research paper deliverables successfully written to work/outputs/.")

Exported top 500 action items to: work/outputs/playbook_ranked_queue.csv
Exported monitoring triggers to: work/outputs/playbook_monitoring_triggers.csv

All research paper deliverables successfully written to work/outputs/.


## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.